In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import random


In [8]:
# Tiny custom dataset
data = [
    "artificial intelligence is transforming the world",
    "deep learning is a part of artificial intelligence",
    "transformers are powerful models",
    "machine learning enables artificial intelligence",
    "artificial intelligence drives innovation"
]

# Tokenizer
def tokenize(text):
    return text.lower().split()

# Build vocab
tokens = [token for sentence in data for token in tokenize(sentence)]
counter = Counter(tokens)
vocab = {word: i+2 for i, word in enumerate(counter)}  # +2 to leave space for <pad>=0, <unk>=1
vocab["<pad>"] = 0
vocab["<unk>"] = 1
inv_vocab = {i: w for w, i in vocab.items()}  # for decoding later

vocab_size = len(vocab)
print("Vocab:", vocab)


Vocab: {'artificial': 2, 'intelligence': 3, 'is': 4, 'transforming': 5, 'the': 6, 'world': 7, 'deep': 8, 'learning': 9, 'a': 10, 'part': 11, 'of': 12, 'transformers': 13, 'are': 14, 'powerful': 15, 'models': 16, 'machine': 17, 'enables': 18, 'drives': 19, 'innovation': 20, '<pad>': 0, '<unk>': 1}


In [9]:
def encode(sentence):
    return [vocab.get(word, vocab["<unk>"]) for word in tokenize(sentence)]

seq_data = [encode(sentence) for sentence in data]

train_X, train_Y = [], []
for seq in seq_data:
    for i in range(1, len(seq)):
        train_X.append(seq[:i])
        train_Y.append(seq[i])

max_len = max(len(x) for x in train_X)

def pad_sequence(seq, max_len):
    return seq + [vocab["<pad>"]] * (max_len - len(seq))

train_X = [pad_sequence(x, max_len) for x in train_X]
train_Y = torch.tensor(train_Y)

train_X = torch.tensor(train_X)
print("Training samples:", train_X.shape)


Training samples: torch.Size([22, 7])


In [10]:
class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, nhead=2, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size)

    def forward(self, src):
        # src: [batch_size, seq_len]
        src = self.embedding(src)  # [batch, seq_len, embed_dim]
        src = src.permute(1, 0, 2)  # Transformer expects [seq_len, batch, embed_dim]
        output = self.transformer(src)
        output = self.fc_out(output[-1])  # Use last token output
        return output


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniTransformer(vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20
for epoch in range(EPOCHS):
    total_loss = 0
    for x, y in zip(train_X, train_Y):
        x, y = x.unsqueeze(0).to(device), y.unsqueeze(0).to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss/len(train_X):.4f}")


C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch [10/20], Loss: 1.2527
Epoch [20/20], Loss: 0.6465


In [12]:
def predict_next_word(model, text, top_k=3):
    model.eval()
    words = tokenize(text)
    seq = [vocab.get(w, vocab["<unk>"]) for w in words]
    seq = pad_sequence(seq, max_len)
    seq = torch.tensor(seq).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(seq)
        probs = torch.softmax(output, dim=-1)
        top_k_probs, top_k_indices = torch.topk(probs, top_k)
    
    predictions = [(inv_vocab[i.item()], p.item()) for i, p in zip(top_k_indices[0], top_k_probs[0])]
    return predictions


In [13]:
prompt = "artificial intelligence"
preds = predict_next_word(model, prompt)
print(f"\nInput: '{prompt}'")
print("Top predictions:")
for word, prob in preds:
    print(f"{word:15s}  ({prob:.4f})")



Input: 'artificial intelligence'
Top predictions:
is               (0.6293)
drives           (0.1832)
enables          (0.0563)
